# OpenHPL SimpleTurbine on Railway
This notebook loads OpenHPL in OpenModelica, simulates `OpenHPL.Examples.SimpleTurbine`, and plots `turbine.Wdot_s` (shaft power).


In [ ]:
import os, subprocess, pathlib, pandas as pd, matplotlib.pyplot as plt
WORK = pathlib.Path('/workspace/run')
WORK.mkdir(parents=True, exist_ok=True)
OPENHPL = pathlib.Path('/workspace/OpenHPL-repo/OpenHPL/package.mo')
OPENIPSL = pathlib.Path('/opt/OpenIPSL/OpenIPSL/package.mo')
print(subprocess.check_output(['omc','--version'], text=True).strip())
print('OpenHPL:', OPENHPL)
print('OpenIPSL:', OPENIPSL, OPENIPSL.exists())


In [ ]:
mos = WORK/'run.mos'
mos.write_text(f'''\nloadModel(Modelica, {{"4.0.0"}});\ngetErrorString();\nloadFile("{OPENIPSL}");\ngetErrorString();\nloadFile("{OPENHPL}");\ngetErrorString();\ncheckModel(OpenHPL.Examples.SimpleTurbine);\ngetErrorString();\nsimulate(OpenHPL.Examples.SimpleTurbine, startTime=0, stopTime=1000, numberOfIntervals=500, tolerance=1e-6, outputFormat="csv", variableFilter="time|control.y|turbine.Wdot_s|turbine.Vdot|turbine.dp");\ngetErrorString();\n''')
r = subprocess.run(['omc', str(mos)], cwd=WORK, text=True, capture_output=True)
print(r.stdout)
print(r.stderr)
if r.returncode != 0:
    raise RuntimeError(f'omc failed with return code {r.returncode}')
csvs = list(WORK.glob('*SimpleTurbine*_res.csv'))
print('Result files:', csvs)
if not csvs:
    raise FileNotFoundError('SimpleTurbine result CSV not found')
RES = csvs[0]


In [ ]:
df = pd.read_csv(RES)
print(df.columns.tolist())
power_col = next(c for c in df.columns if 'turbine.Wdot_s' in c)
out = df[['time', power_col]].rename(columns={power_col:'turbine_Wdot_s_W'})
out['turbine_Wdot_s_MW'] = out['turbine_Wdot_s_W']/1e6
out.to_csv('/workspace/results/turbine_power.csv', index=False)
out.head()


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(out['time'], out['turbine_Wdot_s_MW'])
plt.axvline(500, linestyle='--')
plt.xlabel('Time [s]')
plt.ylabel('Turbine shaft power [MW]')
plt.title('OpenHPL SimpleTurbine: turbine.Wdot_s')
plt.grid(True)
plt.tight_layout()
plt.savefig('/workspace/results/turbine_power.png', dpi=180)
plt.show()


In [ ]:
print('POWER_DATA_START')
for _, row in out.iterrows():
    print(f"{row['time']:.6g},{row['turbine_Wdot_s_MW']:.12g}")
print('POWER_DATA_END')
for t in [0,490,500,515,530,540,1000]:
    i=(out['time']-t).abs().idxmin()
    print(f"sample t={out.loc[i,'time']:.1f}s P={out.loc[i,'turbine_Wdot_s_MW']:.6f} MW")
